# Function CAV, strict labels, PWM/GC/CpG-matched, L2 logistic regression, HepG2 + K562 sites — region 20

The region-8 pipeline (*Function CAV - Strict Labels, PWM-GC-CpG-Matched Ridge, Region 8* Section 1 →
*… L2 Logistic, Region 8* Sections 2–3) for MPRA region 20:

1. **Section 1** (CPU): region 20's SNVs and JASPAR hits, ENCODE ChIP-seq (HepG2 + K562), matched sites per TF. Sites
   already built for region 8 (`fcav_pgc_hk/sites/`) are copied and reused, minus any matched pair with a window
   overlapping region 20; only TFs new to this region are built. Output `fcav_pgc_hk_r20/`.
2. **Section 2** (GPU): every layer, L2 logistic regression (penalty by inner CV), out-of-fold P(bound) and pct per hit.
   Output `fcav_pgc_hk_lr_r20/`.
3. **Section 3** (CPU): AUC by layer; best-p protocol versions B / C and max_abs_signed with gates pct and P(bound),
   shuffled-gate null; the per-position track with each hit's P(bound).

---
# Section 1 — MPRA region, hits, ENCODE, sites *(CPU runtime)*

In [ ]:
!pip install -q pymemesuite pyfaidx requests

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1.1 Settings and the MPRA region

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, shutil
import numpy as np, pandas as pd

ACCESSORY = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/accessary_files"
MPRA_PATH = f"{ACCESSORY}/mpra_sequences_with_embeddings_nt.pkl"   # a .csv with the same columns also works
REGION = 20
CELLS = ["HepG2", "K562"]     # ChIP-seq and DNase from these cells, sites pooled
N_PER_CLASS = 2000          # at most this many sites per class after matching
MATCH_VARS = ["pwm", "gc", "cpg_oe"]
N_MATCH_BINS = 5            # quantile bins per matching variable (5 x 5 x 5 joint bins)
UNBOUND_POOL = 10           # unbound candidates collected: UNBOUND_POOL x N_PER_CLASS, before matching
BOUND_TOP = 0.25            # bound: peaks in the top quarter by signalValue within each experiment
UNBOUND_MIN_DIST = 1000     # unbound: at least this far from every peak of the TF, relaxed peaks included

C = json.load(open(f"{WORK_DIR}/config.json"))
OUT = f"{WORK_DIR}/fcav_pgc_hk_r20"
for d in ("", "/sites"):
    os.makedirs(OUT + d, exist_ok=True)
json.dump({"region": REGION, "cells": CELLS, "n_per_class": N_PER_CLASS, "match_vars": MATCH_VARS, "n_match_bins": N_MATCH_BINS,
           "unbound_pool": UNBOUND_POOL,
           "bound_top": BOUND_TOP,
           "unbound_min_dist": UNBOUND_MIN_DIST}, open(f"{OUT}/settings.json", "w"), indent=1)
TOK, MOFF, WLEN = C["tok"], C["motif_offset"], C["window_len"]
pd.set_option("display.width", 250)
_RC = str.maketrans("ACGT", "TGCA")


def revcomp(s):
    return s.translate(_RC)[::-1]


_m = pd.read_csv(MPRA_PATH, low_memory=False) if MPRA_PATH.endswith(".csv") else pd.read_pickle(MPRA_PATH)
_m = _m[_m["region_id"] == REGION]
_m = _m.drop(columns=[c for c in ("wt_embedding", "mut_embedding", "sequence_mut") if c in _m.columns])
SEQ = _m["sequence"].dropna().iloc[0].upper()
_v = _m[(_m["alt"] != "-") & _m["log2FC"].notna()].copy()
_v["local"] = _v["pos"].astype(int) - _v["start"].astype(int)
_v["ref"], _v["alt"] = _v["ref"].str.upper(), _v["alt"].str.upper()
assert all(0 <= p < len(SEQ) and SEQ[p] == r for p, r in zip(_v["local"], _v["ref"]))
SNV = _v.groupby(["local", "ref", "alt"]).agg(log2FC=("log2FC", "mean"), n_meas=("log2FC", "size")).reset_index()
CHR_HINT = str(_m["chr"].iloc[0]) if "chr" in _m.columns else None
print(f"region {REGION}: {len(SEQ)} bp, {len(SNV)} SNVs")
del _m, _v


In [ ]:
from pymemesuite.common import Sequence, MotifFile
from pymemesuite.fimo import FIMO
from pyfaidx import Fasta

with MotifFile(C["meme_path"]) as _mf:
    _all = list(_mf)
    BACKGROUND = _mf.background
MOTIF = {}
for m in _all:
    MOTIF.setdefault(m.name.decode(), m)           # first matrix with the name, as everywhere else
FIMO_SCAN = FIMO(both_strands=True, threshold=C["fimo_p"])
BASE_IDX = {b: i for i, b in enumerate("ACGT")}


def freq(name):
    f = np.asarray(MOTIF[name].frequencies, float)
    return f / f.sum(1, keepdims=True)


def log_odds(name, strand):
    f = freq(name) if strand == "+" else freq(name)[::-1, [3, 2, 1, 0]]
    return np.log2(np.clip(f, 1e-9, 1.0) / 0.25)


def site_score(seq, name, strand):
    lo = log_odds(name, strand)
    return float(sum(lo[j, BASE_IDX[b]] for j, b in enumerate(seq)))


def scan_many(name, seqs, batch=500):
    out = [[] for _ in seqs]
    for i0 in range(0, len(seqs), batch):
        chunk = [Sequence(s, name=str(i0 + k).encode()) for k, s in enumerate(seqs[i0:i0 + batch])]
        for h in FIMO_SCAN.score_motif(MOTIF[name], chunk, BACKGROUND).matched_elements:
            a, b = sorted((h.start, h.stop))
            out[int(h.source.name.decode())].append((a - 1, b, h.strand, h.pvalue))
    return out


def scan(name, seq):
    return scan_many(name, [seq])[0]


LOCAL_FA = "/content/hg38.fa"
if not os.path.exists(LOCAL_FA):
    print("copying hg38 to local disk (~3 GB, one-off)...")
    shutil.copy(C["genome_fa"], LOCAL_FA)
    if os.path.exists(C["genome_fa"] + ".fai"):
        shutil.copy(C["genome_fa"] + ".fai", LOCAL_FA + ".fai")
GENOME = Fasta(LOCAL_FA, sequence_always_upper=True)
CHROMS = [c for c in [f"chr{i}" for i in range(1, 23)] + ["chrX"] if c in GENOME]
CHROM_LEN = {c: len(GENOME[c]) for c in CHROMS}


def locate(seq, hint):
    # a 40 bp anchor from each end must land at the right distance; <= 2% mismatches allowed
    n, A = len(seq), 40
    order = ([f"chr{hint}".replace("chrchr", "chr")] if hint else []) + CHROMS
    for c in dict.fromkeys(order):
        if c not in GENOME:
            continue
        chrom = GENOME[c][:].seq
        for strand, q in (("+", seq), ("-", revcomp(seq))):
            i = chrom.find(q[:A])
            while i >= 0:
                if chrom[i + n - A:i + n] == q[-A:]:
                    mm = sum(a != b for a, b in zip(chrom[i:i + n], q))
                    if mm <= 0.02 * n:
                        return c, i, strand, mm
                i = chrom.find(q[:A], i + 1)
    return None


loc = locate(SEQ, CHR_HINT)
assert loc, "region not found in hg38"
RCHROM, G0, RSTRAND, _mm = loc
if RSTRAND == "-":
    GSEQ = revcomp(SEQ)
    SNV["local"] = len(SEQ) - 1 - SNV["local"]
    SNV["ref"], SNV["alt"] = SNV["ref"].str.translate(_RC), SNV["alt"].str.translate(_RC)
else:
    GSEQ = SEQ
SNV = SNV.sort_values(["local", "alt"]).reset_index(drop=True)
assert all(GSEQ[p] == r for p, r in zip(SNV["local"], SNV["ref"]))
REGION_IV = (RCHROM, G0, G0 + len(GSEQ))
print(f"region {REGION}: {RCHROM}:{G0}-{G0 + len(GSEQ)} ({RSTRAND} strand), {_mm} mismatches to hg38")


# reuse the sites built for region 8, dropping every matched pair with a window overlapping this region
SHARE = f"{WORK_DIR}/fcav_pgc_hk/sites"
if os.path.isdir(SHARE):
    _c, _s, _e = REGION_IV
    _ov = lambda r: r["chrom"] == _c and r["g_start"] - MOFF < _e and r["g_start"] - MOFF + WLEN > _s
    n_cp, n_drop = 0, 0
    for fn in sorted(os.listdir(SHARE)):
        dst = f"{OUT}/sites/{fn}"
        if not fn.endswith(".json") or os.path.exists(dst):
            continue
        J = json.load(open(f"{SHARE}/{fn}"))
        rec = J["records"]
        if rec:
            nb = sum(r["cls"] == 1 for r in rec)
            assert len(rec) == 2 * nb and all(r["cls"] == 1 for r in rec[:nb])    # bound[i] is paired with unbound[i]
            keep = [i for i in range(nb) if not (_ov(rec[i]) or _ov(rec[nb + i]))]
            n_drop += nb - len(keep)
            J["records"] = [rec[i] for i in keep] + [rec[nb + i] for i in keep]
            if len(keep) < 50:
                J["status"], J["records"] = f"too few sites after dropping region {REGION} overlaps", []
        json.dump(J, open(dst, "w"))
        n_cp += 1
    print(f"reused {n_cp} TFs' sites from region 8; {n_drop} pairs dropped for overlapping region {REGION}")


## 1.2 Every JASPAR hit with SNVs inside, and ΔPWM

In [ ]:
HITS = []
for name in MOTIF:
    for a, b, st, p in scan(name, GSEQ):
        if ((SNV["local"] >= a) & (SNV["local"] < b)).any():
            HITS.append({"motif": name, "start": a, "end": b, "strand": st, "p": float(p),
                         "pwm": site_score(GSEQ[a:b], name, st)})
HITS = pd.DataFrame(HITS).sort_values(["p", "start", "motif"], kind="mergesort").reset_index(drop=True)
HITS["hit_id"] = np.arange(len(HITS))
rows = []
for h in HITS.itertuples():
    lo = log_odds(h.motif, h.strand)
    for i in np.flatnonzero((SNV["local"] >= h.start) & (SNV["local"] < h.end)):
        k = int(SNV.at[i, "local"]) - h.start
        rows.append((i, h.hit_id, lo[k, BASE_IDX[SNV.at[i, "alt"]]] - lo[k, BASE_IDX[SNV.at[i, "ref"]]]))
PAIR = pd.DataFrame(rows, columns=["snv", "hit_id", "dpwm"])
print(f"{len(HITS)} hits of {HITS['motif'].nunique()} motifs contain SNVs; "
      f"{PAIR['snv'].nunique()} of {len(SNV)} SNVs are inside at least one hit")


## 1.3 ENCODE

A motif gets a direction when an ENCODE TF ChIP-seq target in HepG2 or K562 has its name (ignoring case); heterodimer matrices
(`A::B`) are skipped. Bound peaks: one file per experiment (ENCODE's default, else the best IDR set), top `BOUND_TOP`
by signalValue **within that experiment** (signal scales differ between experiments). Exclusion set for unbound sites:
**every** narrowPeak file of the TF in HepG2, the relaxed pre-IDR peaks included.

In [ ]:
import requests

ENC, HDR = "https://www.encodeproject.org", {"accept": "application/json"}
ENC_DIR = f"{WORK_DIR}/encode"                      # shared download cache with the other notebooks
os.makedirs(ENC_DIR, exist_ok=True)
PEAK_COLS = ["chrom", "start", "end", "name", "score", "strand", "signal", "p", "q", "summit"]
PRIORITY = ["conservative IDR thresholded peaks", "optimal IDR thresholded peaks",
            "IDR thresholded peaks", "replicated peaks", "pseudoreplicated peaks"]


def enc_search(**params):
    r = requests.get(f"{ENC}/search/", params={**params, "format": "json", "limit": "all"},
                     headers=HDR, timeout=600)
    if r.status_code == 404:
        return []
    r.raise_for_status()
    return r.json()["@graph"]


def narrowpeak_files(assay_title, cell, target=None):
    q = {"type": "Experiment", "assay_title": assay_title, "status": "released", "biosample_ontology.term_name": cell}
    if target:
        q["target.label"] = target
    rows = []
    for e in enc_search(**q):
        for f in enc_search(type="File", dataset=e["@id"], file_format="bed", assembly="GRCh38", status="released"):
            if str(f.get("file_type", "")).startswith("bed narrowPeak"):
                rows.append({"experiment": e["accession"], "accession": f["accession"],
                             "output_type": f.get("output_type", ""),
                             "preferred_default": bool(f.get("preferred_default", False)), "href": f["href"]})
    return pd.DataFrame(rows)


def fetch_bed(row):
    path = f"{ENC_DIR}/{row['accession']}.bed.gz"
    if not os.path.exists(path):
        with requests.get(ENC + row["href"], stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
        os.rename(path + ".part", path)
    df = pd.read_csv(path, sep="\t", header=None, compression="gzip").iloc[:, :10]
    df.columns = PEAK_COLS[:df.shape[1]]
    return df


def cell_targets(cell):
    exps = enc_search(type="Experiment", assay_title="TF ChIP-seq", status="released",
                      **{"biosample_ontology.term_name": cell}, field="target.label")
    return {e["target"]["label"].upper(): e["target"]["label"] for e in exps if "target" in e}


In [ ]:
def pick_per_experiment(files):
    keep = []
    for _, g in files.groupby("experiment"):
        if g["preferred_default"].any():
            keep.append(g[g["preferred_default"]].iloc[0])
        else:
            rank = g["output_type"].map(lambda t: PRIORITY.index(t) if t in PRIORITY else 99)
            keep.append(g.loc[rank.idxmin()])
    return pd.DataFrame(keep)


def strict_peaks(files):
    # bound: top BOUND_TOP by signal within each experiment; exclusion: every peak file of the TF
    bound = []
    for _, r in pick_per_experiment(files).iterrows():
        d = fetch_bed(r)
        bound.append(d[d["signal"] >= d["signal"].quantile(1 - BOUND_TOP)])
    excl = pd.concat([fetch_bed(r) for _, r in files.iterrows()], ignore_index=True)
    return pd.concat(bound, ignore_index=True), excl


LABEL = {c: cell_targets(c) for c in CELLS}
DN_FILES = {}
for c in CELLS:
    f = narrowpeak_files("DNase-seq", c)
    DN_FILES[c] = f[f["preferred_default"]] if f["preferred_default"].any() else f
    assert len(DN_FILES[c]), f"no DNase-seq in {c}"
SOURCE = {}                                         # motif -> {cell: ChIP files}, or the reason it has none
for m in sorted(HITS["motif"].unique()):
    if "::" in m:
        SOURCE[m] = "dimer matrix"
    elif os.path.exists(f"{OUT}/sites/{m}.json"):
        SOURCE[m] = "cached"
    else:
        fb = {c: narrowpeak_files("TF ChIP-seq", c, LABEL[c][m.upper()]) for c in CELLS if m.upper() in LABEL[c]}
        fb = {c: f for c, f in fb.items() if len(f)}
        SOURCE[m] = fb if fb else f"no ChIP-seq in {' or '.join(CELLS)}"
show = pd.Series({m: (v if isinstance(v, str) else "to build: " + " + ".join(v)) for m, v in SOURCE.items()})
print("  ".join(f"{c}: {len(LABEL[c])} ChIP-seq targets" for c in CELLS))
print(show.value_counts().to_string())


## 1.4 Sites per TF

Per cell with ChIP-seq for the TF. Bound: the best hit (p < `fimo_p`) within `peak_half` of a strict peak's summit, its
centre ≤ `summit_dist` from the summit. Unbound: hits in that cell's DNase peaks ≥ `UNBOUND_MIN_DIST` from every peak
of the TF in **every** cell with data. The two cells' sites are pooled (a locus bound in both is kept once). Windows of 600 bp with the
motif at bp 300, spaced (no two overlap), none overlapping region 20. Then **matched on PWM, GC and CpG o/e**: each cut
into `N_MATCH_BINS` quantile bins over the pooled sites; within each joint bin, bound and unbound sites are paired at
random (the smaller count of the two); at most `N_PER_CLASS` pairs are kept, at random.

In [ ]:
class Intervals:
    def __init__(self, df):
        self.d = {}
        for c, g in df.groupby("chrom"):
            o = np.argsort(g["start"].values)
            s, e = g["start"].values[o], g["end"].values[o]
            new = np.r_[True, s[1:] > np.maximum.accumulate(e)[:-1]]
            idx = np.flatnonzero(new)
            self.d[c] = (s[idx], np.maximum.reduceat(e, idx))

    def dist(self, chrom, pos):
        if chrom not in self.d:
            return np.inf
        st, en = self.d[chrom]
        i = np.searchsorted(st, pos, side="right") - 1
        d = np.inf
        if i >= 0:
            d = 0 if pos < en[i] else pos - en[i] + 1
        if i + 1 < len(st):
            d = min(d, st[i + 1] - pos)
        return d


def gc_frac(s):
    return (s.count("G") + s.count("C")) / len(s)


def cpg_oe(s):
    return s.count("CG") * len(s) / max(s.count("C") * s.count("G"), 1)


def site_record(m, cls, chrom, g_start, g_end, strand, p, signal=np.nan):
    ws = g_start - MOFF
    if ws < 0 or ws + WLEN > CHROM_LEN[chrom]:
        return None
    c, s, e = REGION_IV
    if chrom == c and ws < e and ws + WLEN > s:     # no window may overlap the MPRA region
        return None
    win = GENOME[chrom][ws:ws + WLEN].seq
    if "N" in win:
        return None
    return {"cls": cls, "chrom": chrom, "g_start": int(g_start), "strand": strand, "p": float(p), "signal": float(signal),
            "pwm": site_score(win[MOFF:MOFF + g_end - g_start], m, strand), "window": win}


def spaced(recs):
    kept, occ = [], set()
    for r in sorted(recs, key=lambda r: r["p"]):
        b = (r["chrom"], r["g_start"] // WLEN)
        if any((b[0], b[1] + k) in occ for k in (-1, 0, 1)):
            continue
        kept.append(r)
        occ.add(b)
    return kept


DHS = {}
for c in CELLS:
    _d = pd.concat([fetch_bed(r) for _, r in DN_FILES[c].iterrows()], ignore_index=True)
    _d = _d[_d["chrom"].isin(CHROMS)].sample(frac=1, random_state=C["seed"])
    DHS[c] = list(zip(_d["chrom"], _d["start"].astype(int), _d["end"].astype(int)))
    print(f"{c} DNase peaks: {len(DHS[c]):,}")
del _d


def match_pwm(bound, unbound, rng):
    # joint quantile bins of MATCH_VARS over the pooled sites; one bound to one unbound within each bin
    rec = bound + unbound
    if not bound or not unbound:
        return [], []
    for r in rec:
        r["gc"], r["cpg_oe"] = gc_frac(r["window"]), cpg_oe(r["window"])
    P = pd.DataFrame({v: [r[v] for r in rec] for v in MATCH_VARS})
    P["cls"] = [1] * len(bound) + [0] * len(unbound)
    keys = [pd.qcut(P[v].rank(method="first"), N_MATCH_BINS, labels=False) for v in MATCH_VARS]
    pairs = []
    for _, g in P.groupby(keys):
        b = rng.permutation(np.array(g.index[g["cls"] == 1]))
        u = rng.permutation(np.array(g.index[g["cls"] == 0]))
        k = min(len(b), len(u))
        pairs += list(zip(b[:k], u[:k]))
    pairs = [pairs[i] for i in rng.permutation(len(pairs))[:N_PER_CLASS]]
    return [rec[i] for i, _ in pairs], [rec[j] for _, j in pairs]


def spans(m):
    w = len(freq(m))
    return w, MOFF // TOK, -(-(MOFF + w) // TOK)


def build(m, files_by_cell):
    t0 = time.time()
    w = len(freq(m))
    peaks = {c: strict_peaks(f) for c, f in files_by_cell.items()}
    excl = Intervals(pd.concat([e for _, e in peaks.values()], ignore_index=True))   # every peak, every cell
    bound, unbound, n_pk = [], [], {}
    for c, (bpk, _) in peaks.items():
        pk = bpk[bpk["chrom"].isin(CHROMS)].copy()
        pk["abs_summit"] = np.where(pk["summit"] >= 0, pk["start"] + pk["summit"], (pk["start"] + pk["end"]) // 2)
        pk = pk.sort_values("signal", ascending=False).drop_duplicates(["chrom", "abs_summit"])
        pk = pk[(pk["abs_summit"] >= C["peak_half"]) & (pk["abs_summit"] + C["peak_half"] <= pk["chrom"].map(CHROM_LEN))]
        n_pk[c] = len(pk)
        chroms, summits, sig = pk["chrom"].tolist(), pk["abs_summit"].astype(int).tolist(), pk["signal"].tolist()
        seqs = [GENOME[ch][s - C["peak_half"]:s + C["peak_half"]].seq for ch, s in zip(chroms, summits)]
        for ch, s, sg, hits in zip(chroms, summits, sig, scan_many(m, seqs)):
            if not hits:
                continue
            a, b, st, p = min(hits, key=lambda h: h[3])
            if abs((a + b) / 2 - C["peak_half"]) > C["summit_dist"] or b - a != w:
                continue
            r = site_record(m, 1, ch, s - C["peak_half"] + a, s - C["peak_half"] + b, st, p, sg)
            if r:
                r["cell"] = c
                bound.append(r)
        want, got = UNBOUND_POOL * N_PER_CLASS // len(peaks), 0
        for i0 in range(0, len(DHS[c]), 2000):
            if got >= want:
                break
            chunk = DHS[c][i0:i0 + 2000]
            for (ch, s, e), hits in zip(chunk, scan_many(m, [GENOME[ch][s:e].seq for ch, s, e in chunk])):
                for a, b, st, p in hits:
                    if b - a != w or excl.dist(ch, s + (a + b) // 2) < UNBOUND_MIN_DIST:
                        continue
                    r = site_record(m, 0, ch, s + a, s + b, st, p)
                    if r:
                        r["cell"] = c
                        unbound.append(r)
                        got += 1
    bound, unbound = spaced(bound), spaced(unbound)
    rng = np.random.default_rng(C["seed"])
    nb, nu = len(bound), len(unbound)
    bound, unbound = match_pwm(bound, unbound, rng)
    rec = bound + unbound
    for r in rec:
        r["gc"], r["cpg_oe"] = gc_frac(r["window"]), cpg_oe(r["window"])
    w, K0, K1 = spans(m)
    ok = min(len(bound), len(unbound)) >= 50
    json.dump({"tf": m, "status": "built" if ok else f"too few sites (bound {nb}, unbound {nu})", "width": w,
               "K0": K0, "K1": K1, "pool_bound": nb, "pool_unbound": nu, "records": rec if ok else []},
              open(f"{OUT}/sites/{m}.json", "w"))
    by_cell = "  ".join(f"{c} {sum(r['cell'] == c for r in bound)}/{sum(r['cell'] == c for r in unbound)}" for c in CELLS)
    print(f"{m:10s} strict peaks {'+'.join(str(n_pk.get(c, 0)) for c in CELLS):>13s}  pool bound {nb:5d} unbound {nu:5d}"
          f" -> matched {len(bound):5d} per class (bound/unbound by cell: {by_cell})"
          f"{'' if ok else '  TOO FEW'}  [{time.time() - t0:.0f}s]", flush=True)


todo = [m for m, v in SOURCE.items() if not isinstance(v, str)]
print(f"{len(CHROMS)} chromosomes; {len(todo)} TFs to build")
t_all = time.time()
for k, m in enumerate(todo):
    print(f"[{k + 1}/{len(todo)}] ", end="")
    build(m, SOURCE[m])
print(f"done [{(time.time() - t_all) / 60:.1f} min]")


## 1.5 MPRA windows for every hit of a motif with a direction

In [ ]:
STATUS = {}
for m, v in SOURCE.items():
    STATUS[m] = json.load(open(f"{OUT}/sites/{m}.json"))["status"] if not isinstance(v, str) or v == "cached" else v
HAVE = sorted(m for m, s in STATUS.items() if s == "built")
MW = []
for h in HITS[HITS["motif"].isin(HAVE)].itertuples():
    w, K0, K1 = spans(h.motif)
    assert h.end - h.start == w
    ws = G0 + h.start - MOFF
    win = list(GENOME[RCHROM][ws:ws + WLEN].seq)
    for k in range(max(0, G0 - ws), min(WLEN, G0 + len(GSEQ) - ws)):
        win[k] = GSEQ[ws - G0 + k]                   # the MPRA bases inside the region
    win = "".join(win)
    assert len(win) == WLEN and "N" not in win and win[MOFF:MOFF + w] == GSEQ[h.start:h.end]
    MW.append({"hit_id": int(h.hit_id), "motif": h.motif, "start": int(h.start), "end": int(h.end), "strand": h.strand,
               "p": float(h.p), "pwm": float(h.pwm), "K0": K0, "K1": K1, "window": win})
SNV.to_csv(f"{OUT}/snvs.csv", index=False)
HITS.to_csv(f"{OUT}/hits.csv", index=False)
PAIR.to_csv(f"{OUT}/snv_hit_dpwm.csv", index=False)
json.dump({"hits": MW, "status": STATUS, "region": REGION, "region_iv": REGION_IV}, open(f"{OUT}/mpra_windows.json", "w"),
          default=float)
cov = PAIR[PAIR["hit_id"].isin([h["hit_id"] for h in MW])]["snv"].nunique()
print(f"{len(HAVE)} TFs with a direction; {len(MW)} hit windows; SNVs inside a hit with a direction: {cov} "
      f"(any hit: {PAIR['snv'].nunique()}; all SNVs: {len(SNV)})")
print("\nmotifs without a direction:")
print(pd.Series({m: s for m, s in STATUS.items() if s != "built"}).to_string())


## 1.6 End of Section 1

In [ ]:
from google.colab import runtime
runtime.unassign()

---
# Section 2 — embed every layer and fit *(GPU runtime)*

Per TF: the sites' motif tokens `[K0, K1)` at every hidden state and the MPRA hits' windows the same way. 5 stratified
folds over sites; in each fold the features are standardised on the training sites, the penalty is chosen by 3-fold CV
inside the training sites (no test data involved), and the model refitted on all training sites with that penalty. Saved
per TF (`fits/<TF>.npz`): out-of-fold logit and AUC per layer, the chosen penalties, each MPRA hit's out-of-fold
probability, logit and pct per layer, and the direction fitted on all sites (`w_all`, `mu`, `sd`; penalty = the median
of the folds' choices). The embeddings are not stored.

In [ ]:
!pip install -q -U "transformers==4.57.6" "tokenizers==0.22.0" "huggingface_hub==0.34.0" "safetensors==0.4.5"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, gc
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForMaskedLM

C = json.load(open(f"{WORK_DIR}/config.json"))
SRC = f"{WORK_DIR}/fcav_pgc_hk_r20"                   # sites and MPRA windows (Section 1 of the ridge notebook)
OUT = f"{WORK_DIR}/fcav_pgc_hk_lr_r20"
os.makedirs(f"{OUT}/fits", exist_ok=True)
MWJ = json.load(open(f"{SRC}/mpra_windows.json"))
TOK = C["tok"]
DEV = torch.device("cuda")
TOKZ = AutoTokenizer.from_pretrained(C["model_name"], trust_remote_code=True)
MODEL = AutoModelForMaskedLM.from_pretrained(C["model_name"], trust_remote_code=True).to(DEV).eval()
HID, NL = MODEL.config.hidden_size, MODEL.config.num_hidden_layers
LAYERS = list(range(NL + 1))
_ids = TOKZ("A" * C["window_len"])["input_ids"]
LEAD = int(_ids[0] == TOKZ.cls_token_id)
TRAIL = int(TOKZ.eos_token_id is not None and _ids[-1] == TOKZ.eos_token_id)
assert len(_ids) - LEAD - TRAIL == C["window_len"] // TOK
print(f"{C['model_name']}: layers 0-{NL}, hidden {HID}")


@torch.no_grad()
def embed(seqs, ranges, batch=64):
    out = np.zeros((len(seqs), len(LAYERS), HID), np.float16)
    for i in range(0, len(seqs), batch):
        ids = TOKZ(seqs[i:i + batch], return_tensors="pt")["input_ids"].to(DEV)
        assert ids.shape[1] == LEAD + C["window_len"] // TOK + TRAIL
        S = torch.stack(MODEL(input_ids=ids, output_hidden_states=True).hidden_states, 1)   # (B, layers, T, H)
        rg = ranges[i:i + batch]
        if len(set(rg)) == 1:                                              # one token range for the whole batch
            k0, k1 = rg[0]
            E = S[:, :, LEAD + k0:LEAD + k1].mean(2)
        else:
            E = torch.stack([S[j, :, LEAD + k0:LEAD + k1].mean(1) for j, (k0, k1) in enumerate(rg)])
        out[i:i + len(rg)] = E.half().cpu().numpy()
        del S, E
    return out


if DEV.type == "cuda":
    torch.backends.cuda.preferred_linalg_library("cusolver")   # batched linalg.solve via cuBLAS fails on some Colab builds
LAMS = torch.logspace(4, -1, 11, device=DEV)      # L2 penalties tried, strongest first (standardised features, summed log-loss)
NEWTON = 15                                       # max Newton steps per penalty
INNER = 3                                         # inner CV folds for the penalty


def logit_path(X, y):
    # L2 logistic regression for every layer at once and every penalty in LAMS (warm-started, strongest first).
    # X: (layers, n, d) standardised; y: (n,) 0/1. Returns W (lams, layers, d + 1) = [weights, intercept].
    nl, n, d = X.shape
    Xa = torch.cat([X, torch.ones(nl, n, 1, device=X.device)], 2)
    pen = torch.ones(d + 1, device=X.device); pen[-1] = 0                # intercept not penalised
    eye = torch.eye(d + 1, device=X.device)
    w = torch.zeros(nl, d + 1, device=X.device)
    w[:, -1] = torch.logit(y.mean().clamp(1e-3, 1 - 1e-3))

    def loss(w, lam):
        z = torch.einsum("lnd,ld->ln", Xa, w)
        return F.binary_cross_entropy_with_logits(z, y.expand_as(z), reduction="none").sum(1) + lam / 2 * (pen * w * w).sum(1)

    out = []
    for lam in LAMS.tolist():
        for _ in range(NEWTON):
            z = torch.einsum("lnd,ld->ln", Xa, w)
            p = torch.sigmoid(z)
            g = torch.einsum("lnd,ln->ld", Xa, p - y) + lam * pen * w
            H = (Xa * (p * (1 - p))[..., None]).transpose(1, 2) @ Xa + torch.diag(lam * pen + 1e-6)
            Lc, _ = torch.linalg.cholesky_ex(H)                              # H is SPD (penalty > 0, intercept term 1e-6)
            step = torch.cholesky_solve(g[..., None], Lc)[..., 0]
            ts = torch.tensor([1, .5, .25, .125, 0], device=X.device)
            L = torch.stack([loss(w - t * step, lam) for t in ts])         # (steps, layers); t = 0 keeps w
            t = ts[L.argmin(0)][:, None]
            w = w - t * step
            if (t * step).abs().max() < 1e-4:
                break
        out.append(w.clone())
    return torch.stack(out)


def bce(Xte, yte, W):
    # held-out summed log-loss of every (penalty, layer); Xte (layers, n, d), W (lams, layers, d + 1)
    z = torch.einsum("lnd,kld->kln", Xte, W[..., :-1]) + W[..., -1:]
    return F.binary_cross_entropy_with_logits(z, yte.expand_as(z), reduction="none").sum(2)


def fit_logit(X, y, seed):
    # penalty per layer by INNER-fold CV log-loss on these sites, then the model on all of them at that penalty.
    # Returns W (layers, d + 1), the chosen penalty index per layer, and the whole path.
    ll = torch.zeros(len(LAMS), X.shape[0], device=X.device)
    yn = y.cpu().numpy()
    for itr, ite in StratifiedKFold(INNER, shuffle=True, random_state=seed).split(np.zeros(len(yn)), yn):
        itr, ite = torch.tensor(itr, device=X.device), torch.tensor(ite, device=X.device)
        ll += bce(X[:, ite], y[ite], logit_path(X[:, itr], y[itr]))
    k = ll.argmin(0)
    Wp = logit_path(X, y)
    return Wp[k, torch.arange(X.shape[0])], k.cpu().numpy(), Wp


def pct_of(ref, v):
    # percentile of each v among ref (ties count half), 0-100
    ref = np.sort(ref)
    lo, hi = np.searchsorted(ref, v, "left"), np.searchsorted(ref, v, "right")
    return 100 * (lo + hi) / 2 / len(ref)


In [ ]:
H = MWJ["hits"]
built = sorted(m for m, s in MWJ["status"].items() if s == "built")
t_all = time.time()
for k, m in enumerate(built):
    p = f"{OUT}/fits/{m}.npz"
    if os.path.exists(p):
        continue
    t0 = time.time()
    J = json.load(open(f"{SRC}/sites/{m}.json"))
    R = J["records"]
    y = np.array([r["cls"] for r in R])
    X = embed([r["window"] for r in R], [(J["K0"], J["K1"])] * len(R))
    hi = [h for h in H if h["motif"] == m]
    XH = embed([h["window"] for h in hi], [(h["K0"], h["K1"]) for h in hi])
    folds = list(StratifiedKFold(5, shuffle=True, random_state=C["seed"]).split(X, y))
    Xg = torch.tensor(X, dtype=torch.float32, device=DEV).transpose(0, 1).contiguous()     # (layers, n, H)
    Hg = torch.tensor(XH, dtype=torch.float32, device=DEV).transpose(0, 1).contiguous()    # (layers, hits, H)
    yg = torch.tensor(y, dtype=torch.float32, device=DEV)
    oof = np.zeros((len(y), len(LAYERS)), np.float32)
    pk, dk, qk, kk = [], [], [], []
    for f, (tr, te) in enumerate(folds):
        mu, sd = Xg[:, tr].mean(1, keepdim=True), Xg[:, tr].std(1, keepdim=True) + 1e-6
        W, ki, _ = fit_logit((Xg[:, tr] - mu) / sd, yg[tr], C["seed"] + f)
        kk.append(ki)
        dte = (torch.einsum("lnd,ld->ln", (Xg[:, te] - mu) / sd, W[:, :-1]) + W[:, -1:]).cpu().numpy()   # (layers, te) logits
        dh = (torch.einsum("lnd,ld->ln", (Hg - mu) / sd, W[:, :-1]) + W[:, -1:]).cpu().numpy()          # (layers, hits) logits
        oof[te] = dte.T
        pk.append(np.stack([pct_of(dte[li], dh[li]) for li in range(len(LAYERS))], 1))
        dk.append(dh.T)
        qk.append(1 / (1 + np.exp(-dh.T)))
    pct = np.mean(pk, 0).astype(np.float32)
    auc = np.array([roc_auc_score(y, oof[:, li]) for li in range(len(LAYERS))])
    kk = np.array(kk)                                                     # (folds, layers) penalty index
    mu, sd = Xg.mean(1, keepdim=True), Xg.std(1, keepdim=True) + 1e-6
    Wp = logit_path((Xg - mu) / sd, yg)
    kall = np.median(kk, 0).round().astype(int)
    W = Wp[torch.tensor(kall, device=DEV), torch.arange(len(LAYERS), device=DEV)]
    lam = LAMS.cpu().numpy()
    np.savez(p, auc=auc, oof=oof, cls=y, pct=pct, prob_hit=np.mean(qk, 0).astype(np.float32),
             dec_hit=np.mean(dk, 0).astype(np.float32), lam=lam[kk], lam_all=lam[kall],
             w_all=W.cpu().numpy(), mu=mu[:, 0].cpu().numpy(), sd=sd[:, 0].cpu().numpy(),
             hit_id=np.array([h["hit_id"] for h in hi]), layers=np.array(LAYERS))
    print(f"[{k + 1}/{len(built)}] {m:10s} sites {len(y):5d}  hits {len(hi):2d}  AUC L0 {auc[0]:.3f}  "
          f"best L{int(auc.argmax())} {auc.max():.3f}  last {auc[-1]:.3f}  log10 λ median {np.median(np.log10(lam[kk])):.1f} "
          f"(grid -1..4)  [{time.time() - t0:.0f}s, {(time.time() - t_all) / 60:.1f} min total]", flush=True)
    del X, XH, Xg, Hg, Wp; gc.collect(); torch.cuda.empty_cache()
print("done")


## 2.1 End of Section 2

In [ ]:
from google.colab import runtime
runtime.unassign()

---
# Section 3 — analysis *(CPU runtime)*

In [ ]:
!pip install -q scikit-learn scipy pandas matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3.1 AUC of the function direction at every layer

Out-of-fold AUC, bound vs unbound. References: the AUC of the site's PWM score, GC and CpG o/e alone (~0.5 after
matching). The ridge notebook's mean AUC is printed for comparison. Last block: TFs with ≥ `MIN_PAIRS` matched pairs vs the rest.

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

C = json.load(open(f"{WORK_DIR}/config.json"))
SRC = f"{WORK_DIR}/fcav_pgc_hk_r20"                   # sites, hits, SNVs (ridge notebook, Section 1)
OUT = f"{WORK_DIR}/fcav_pgc_hk_lr_r20"
MWJ = json.load(open(f"{SRC}/mpra_windows.json"))
LP = C["primary_layer"]
pd.set_option("display.width", 250)
INK = "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": INK,
                     "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK})
TFS = sorted(m for m, s in MWJ["status"].items() if s == "built" and os.path.exists(f"{OUT}/fits/{m}.npz"))
FIT = {m: np.load(f"{OUT}/fits/{m}.npz") for m in TFS}
LAYERS = list(FIT[TFS[0]]["layers"])
rows = []
for m in TFS:
    R = pd.DataFrame(json.load(open(f"{SRC}/sites/{m}.json"))["records"])
    assert (R["cls"].values == FIT[m]["cls"]).all()
    row = {"motif": m, "bound": int(R.cls.sum()), "unbound": int((R.cls == 0).sum()),
           "bound HepG2": int(((R.cls == 1) & (R.cell == "HepG2")).sum()), "bound K562": int(((R.cls == 1) & (R.cell == "K562")).sum())}
    for v in ("pwm", "gc", "cpg_oe"):
        row[f"AUC {v}"] = roc_auc_score(R["cls"], R[v])
    row.update({l: a for l, a in zip(LAYERS, FIT[m]["auc"])})
    rows.append(row)
A = pd.DataFrame(rows).set_index("motif")
A.to_csv(f"{OUT}/auc_by_layer.csv")
show = ["bound", "bound HepG2", "bound K562", "AUC pwm", "AUC gc", "AUC cpg_oe"] + [l for l in LAYERS if l % 3 == 0 or l == LAYERS[-1]]
print(A[show].to_string(float_format=lambda x: f"{x:.3f}"))
mean = A[LAYERS].mean()
print(f"\nmean over {len(A)} TFs: " + "  ".join(f"L{l} {mean[l]:.3f}" for l in LAYERS if l % 3 == 0 or l == LAYERS[-1]))
print(f"best layer (mean) {mean.idxmax()} ({mean.max():.3f}); layer 0 {mean[0]:.3f}; last {mean[LAYERS[-1]]:.3f}; "
      f"references: PWM {A['AUC pwm'].mean():.3f}, GC {A['AUC gc'].mean():.3f}, CpG o/e {A['AUC cpg_oe'].mean():.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
for m in TFS:
    ax.plot(LAYERS, A.loc[m, LAYERS], c="#c4c4c4", lw=.7)
ax.plot(LAYERS, mean, c="#2a78d6", lw=2.2, marker="o", ms=3, label=f"mean over {len(TFS)} TFs")
for v, c in (("pwm", "#eb6834"), ("gc", "#2fa37a")):
    ax.axhline(A[f"AUC {v}"].mean(), c=c, ls="--", lw=1, label=f"{v} alone (mean)")
ax.axhline(0.5, c=INK, lw=.6, ls=":")
ax.set_xlabel("layer"); ax.set_ylabel("AUC bound vs unbound (out of fold)")
ax.set_title("function direction, strict labels, PWM/GC/CpG-matched, L2 logistic, HepG2 + K562 sites", loc="left", fontsize=10)
ax.legend(frameon=False, fontsize=8)
if os.path.exists(f"{SRC}/auc_by_layer.csv"):
    AR = pd.read_csv(f"{SRC}/auc_by_layer.csv", index_col="motif")
    AR = AR.loc[AR.index.intersection(A.index), [str(l) for l in LAYERS]].mean()
    print("ridge (same sites):  " + "  ".join(f"L{l} {AR[str(l)]:.3f}" for l in LAYERS if l % 3 == 0 or l == LAYERS[-1]))
    ax.plot(LAYERS, AR.values, c="#2a78d6", lw=1, ls="--", label="ridge, same sites (mean)")
    ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/auc_by_layer.png", dpi=150, bbox_inches="tight"); plt.show()

MIN_PAIRS = 1000
BIG = A.index[A["bound"] >= MIN_PAIRS].tolist()
for name, ms in (("big", BIG), ("rest", [m for m in A.index if m not in BIG])):
    if ms:
        mu = A.loc[ms, LAYERS].mean()
        print(f"AUC {name:4s} ({len(ms):2d} TFs, ≥ {MIN_PAIRS} pairs: {name == 'big'}): "
              + "  ".join(f"L{l} {mu[l]:.3f}" for l in LAYERS if l % 3 == 0 or l == LAYERS[-1]) + f" | best L{mu.idxmax()} {mu.max():.3f}")


## 3.2 Region 20, best-p protocol (main)

One hit per SNV, the one with the smallest FIMO p (the gate never chooses the hit). **B**: SNVs whose best-p hit among
all hits has a direction; **C**: SNVs in at least one hit with a direction, best-p hit among those. Gates: pct/100 and
P(bound) (out-of-fold logistic probability). Spearman, signed (with log2FC) and |·| (with |log2FC|). Null: the gate
shuffled across hits, 1000 times at the primary layer, 200 elsewhere.

In [ ]:
SNV = pd.read_csv(f"{SRC}/snvs.csv")
PAIR = pd.read_csv(f"{SRC}/snv_hit_dpwm.csv")
HITS = pd.read_csv(f"{SRC}/hits.csv").set_index("hit_id")
assert HITS["p"].is_monotonic_increasing          # hit_id is in FIMO-p order, so the best-p hit is the smallest id
GATES = {"pct": pd.concat([pd.DataFrame(FIT[m]["pct"] / 100, index=FIT[m]["hit_id"], columns=LAYERS) for m in TFS]),
         "prob": pd.concat([pd.DataFrame(FIT[m]["prob_hit"], index=FIT[m]["hit_id"], columns=LAYERS) for m in TFS])}
HAVE = GATES["pct"].index
best_all = PAIR.groupby("snv")["hit_id"].min()
VER = {"B": best_all[best_all.isin(HAVE)],
       "C": PAIR[PAIR["hit_id"].isin(HAVE)].groupby("snv")["hit_id"].min()}
DPWM = PAIR.set_index(["snv", "hit_id"])["dpwm"]


def rho(dp, idx, yy, g):
    pred = dp * g[idx]
    if np.std(pred) == 0:
        return np.nan, np.nan
    return stats.spearmanr(pred, yy)[0], stats.spearmanr(np.abs(pred), np.abs(yy))[0]


rng = np.random.default_rng(0)
rows = []
for v, hit_of in VER.items():
    hs, idx = np.unique(hit_of.values, return_inverse=True)
    dp = DPWM.loc[list(zip(hit_of.index, hit_of.values))].values
    yy = SNV.loc[hit_of.index, "log2FC"].values
    none = rho(dp, idx, yy, np.ones(len(hs)))
    for gname, G_ in GATES.items():
        for l in LAYERS:
            gl = G_.loc[hs, l].values
            obs = rho(dp, idx, yy, gl)
            null = np.array([rho(dp, idx, yy, rng.permutation(gl)) for _ in range(1000 if l == LP else 200)])
            rows.append({"version": v, "gate": gname, "layer": l, "SNVs": len(hit_of), "hits": len(hs),
                         "none signed": none[0], "signed": obs[0], "null signed": np.nanmean(null[:, 0]),
                         "p signed": np.mean(null[:, 0] >= obs[0]),
                         "none |.|": none[1], "|.|": obs[1], "null |.|": np.nanmean(null[:, 1]),
                         "p |.|": np.mean(null[:, 1] >= obs[1]), "gate sd": gl.std()})
BP = pd.DataFrame(rows)
BP.to_csv(f"{OUT}/region20_bestp_by_layer.csv", index=False)
show = [l for l in LAYERS if l % 3 == 0 or l == LAYERS[-1] or l == LP]
for v in VER:
    for gname in GATES:
        t = BP[(BP.version == v) & (BP.gate == gname)].set_index("layer")
        print(f"=== version {v}, gate {gname}: {t['SNVs'].iloc[0]} SNVs, {t['hits'].iloc[0]} hits; no gate: signed "
              f"{t['none signed'].iloc[0]:+.3f}, |.| {t['none |.|'].iloc[0]:+.3f}")
        print(t.drop(columns=["version", "gate", "SNVs", "hits", "none signed", "none |.|"]).loc[show]
              .to_string(float_format=lambda x: f"{x:+.3f}"))
        print(f"layers with p < 0.05: signed {int((t['p signed'] < .05).sum())} of {len(t)}, |.| {int((t['p |.|'] < .05).sum())}\n")

fig, ax = plt.subplots(2, 2, figsize=(13, 7.5), sharex=True)
for i, v in enumerate(VER):
    for j, k in enumerate(("signed", "|.|")):
        a = ax[i, j]
        for gname, c in zip(GATES, ("#2a78d6", "#eb6834")):
            t = BP[(BP.version == v) & (BP.gate == gname)]
            a.plot(t.layer, t[k], marker="o", ms=3, c=c, label=f"ΔPWM × {gname}")
            a.plot(t.layer, t[f"null {k}"], c=c, lw=.8, ls=":")
        a.axhline(t[f"none {k}"].iloc[0], c=INK, lw=.8, ls="--", label="no gate")
        a.axvline(LP, c=INK, lw=.4, ls="--")
        a.set_title(f"version {v}, Spearman {k} (dotted: shuffled)", loc="left", fontsize=10)
        a.legend(frameon=False, fontsize=8)
for a in ax[1]:
    a.set_xlabel("layer")
plt.tight_layout(); plt.savefig(f"{OUT}/region20_bestp_by_layer.png", dpi=150, bbox_inches="tight"); plt.show()


## 3.3 Region 20, max_abs_signed

SNVs inside at least one hit with a direction; per SNV the hit with the largest |ΔPWM × gate| (sign kept). Pearson and
Spearman with log2FC; null: the gate shuffled across hits (200 per layer, 1000 at the primary layer).

In [ ]:
P = PAIR[PAIR["hit_id"].isin(HAVE)].reset_index(drop=True)
y = SNV.loc[np.sort(P["snv"].unique()), "log2FC"]


def max_abs(val):
    d = P.assign(v=val, a=np.abs(val)).sort_values(["snv", "a", "hit_id"], ascending=[True, False, True])
    return d.drop_duplicates("snv").set_index("snv")["v"].reindex(y.index)


def corr(pred):
    return stats.pearsonr(pred, y)[0], stats.spearmanr(pred, y)[0]


base = corr(max_abs(P["dpwm"].values))
hits = HAVE.values
rows = []
for gname, G_ in GATES.items():
    for l in LAYERS:
        obs = corr(max_abs(P["dpwm"].values * P["hit_id"].map(G_[l]).values))
        null = np.array([corr(max_abs(P["dpwm"].values * P["hit_id"].map(dict(zip(hits, rng.permutation(G_.loc[hits, l].values)))).values))
                         for _ in range(1000 if l == LP else 200)])
        rows.append({"gate": gname, "layer": l, "Pearson": obs[0], "null P": null[:, 0].mean(), "p P": np.mean(null[:, 0] >= obs[0]),
                     "Spearman": obs[1], "null S": null[:, 1].mean(), "p S": np.mean(null[:, 1] >= obs[1])})
G = pd.DataFrame(rows)
G.to_csv(f"{OUT}/region20_maxabs_by_layer.csv", index=False)
print(f"{P.snv.nunique()} SNVs inside {P.hit_id.nunique()} hits; ΔPWM alone: Pearson {base[0]:+.3f}, Spearman {base[1]:+.3f}")
for gname in GATES:
    t = G[G.gate == gname].set_index("layer").drop(columns="gate")
    print(f"\n=== gate {gname}")
    print(t.loc[show].to_string(float_format=lambda x: f"{x:+.3f}"))
    print(f"layers with p < 0.05: Pearson {int((t['p P'] < .05).sum())} of {len(t)}, Spearman {int((t['p S'] < .05).sum())}")


## 3.4 Track: log2FC per position and P(bound) of every hit with a direction

Top: mean log2FC per position (grey: outside every hit with a direction). Bottom: each hit, coloured and labelled by
its out-of-fold P(bound) at `L_SHOW`; outlined = used by version B.

In [ ]:
import matplotlib.colors as mcolors
L_SHOW = 24                                         # layer whose P(bound) is shown
C1, C2, GREY = "#2a78d6", "#eb6834", "#c4c4c4"
cm, nm = plt.get_cmap("RdBu_r"), mcolors.TwoSlopeNorm(vmin=0, vcenter=0.5, vmax=1)
PROB = GATES["prob"][L_SHOW]
used_B = set(VER["B"].values)

pos = SNV.groupby("local")["log2FC"].mean()
covered = set(SNV.loc[PAIR.loc[PAIR["hit_id"].isin(PROB.index), "snv"].unique(), "local"])
cov = np.array([p in covered for p in pos.index])
hh = HITS.loc[PROB.index, ["motif", "start", "end"]].copy()
hh["hit_id"] = PROB.index.values
hh = hh.sort_values(["start", "end"]).reset_index(drop=True)
hh["prob"] = hh["hit_id"].map(PROB)
hh["label"] = [f"{m} {p:.2f}" for m, p in zip(hh["motif"], hh["prob"])]

span = pos.index.max() - pos.index.min() + 1
ends, lane = [], []
for a, b, lab in zip(hh["start"], hh["end"], hh["label"]):
    b = b + 1 + 0.0045 * span * len(lab)             # room for the label
    k = next((i for i, e in enumerate(ends) if e <= a), len(ends))
    if k == len(ends):
        ends.append(b)
    else:
        ends[k] = b
    lane.append(k)
nrow = max(lane) + 1 if lane else 1

fig, ax = plt.subplots(2, 1, figsize=(15, 3 + 0.32 * nrow), sharex=True,
                       gridspec_kw={"height_ratios": [3, 0.32 * nrow + 0.4]})
ax[0].bar(pos.index, pos.values, width=1, color=np.where(~cov, GREY, np.where(pos.values < 0, C1, C2)))
ax[0].axhline(0, color=INK, lw=.6); ax[0].set_ylabel("mean log2FC per position")
ax[0].set_title(f"region {MWJ['region']}: {len(hh)} hits with a direction (grey bars: outside all of them; outlined: used by "
                f"version B); colour = P(bound), L{L_SHOW}", loc="left", fontsize=10)
for (_, r), k in zip(hh.iterrows(), lane):
    inB = r["hit_id"] in used_B
    ax[1].add_patch(plt.Rectangle((r["start"], k), r["end"] - r["start"], 0.8, facecolor=cm(nm(r["prob"])),
                                  edgecolor="black" if inB else "none", linewidth=1.0 if inB else 0))
    ax[1].text(r["end"] + 0.5, k + 0.4, r["label"], va="center", fontsize=6, color=INK)
ax[1].set_ylim(-0.2, nrow); ax[1].set_yticks([])
ax[1].set_xlabel(f"position in region {MWJ['region']}")
fig.colorbar(plt.cm.ScalarMappable(norm=nm, cmap=cm), ax=ax, shrink=.6, label=f"P(bound), L{L_SHOW}")
plt.savefig(f"{OUT}/region{MWJ['region']}_track_prob_L{L_SHOW}.png", dpi=150, bbox_inches="tight"); plt.show()


## 3.5 End

In [ ]:
from google.colab import runtime
runtime.unassign()